# 2b — $\{f, g\} = \omega(X_f, X_g) = X_f(g) = \iota_{X_f} dg = \iota_{X_f}\iota_{X_g}\omega$

**Problem (b).** Aynı symplectic kurulumda ($d\omega = 0$, $\iota_{X_f}\omega = df$, $\{f,g\} := \pi(df, dg)$, $\pi = \omega^{-1}$) beş-terimli zinciri kapat:

$$
\{f, g\}
\;\underset{(1)}{=}\; \omega(X_f, X_g)
\;\underset{(2)}{=}\; X_f(g)
\;\underset{(3)}{=}\; \iota_{X_f} dg
\;\underset{(4)}{=}\; \iota_{X_f}\iota_{X_g}\omega.
$$

Beş terim bir zincir oluşturduğu için uçtan uca tek eşitlik: $\{f, g\} = X_f(g)$. Notebook bunu kapatıyor ve ara terimler **ispat zincirindeki ara değerler olarak** otomatik görünüyor.

## Zincirdeki eşitliklerin statüsü

Sistemimiz şu anda form'u **grading seviyesinde** (derece-2 graded sembol) biliyor; `ω(X, Y)` gibi intrinsik 2-form evaluation node'u **yok** (bkz. Faz 12 taslağı). Bu yüzden zincirdeki dört eşitliğin durumu farklı:

| # | Eşitlik | Statü | Kaynak |
|---|---|---|---|
| (1) | $\{f,g\} = \omega(X_f, X_g)$ | Tanımsal (problem given: $\pi = \omega^{-1}$) | Inline aksiyom A1 |
| (2) | $\omega(X_f, X_g) = \iota_{X_f}\iota_{X_g}\omega$ | Tanımsal (2-form evaluation konvansiyonu) | Inline aksiyom A2 |
| (3) | $\iota_{X_g}\omega = dg$ | Hamiltonian defining relation | Inline aksiyom A3 (2a'nın $g$ muadili) |
| (4) | $\iota_X(df) = X(f)$ | **Yerleşik pairing kuralı** | `IotaOnExactOneFormDefinition` (default engine) |

Faz 12 landing ettiğinde (1) ve (2) **aksiyom** olmaktan çıkıp **theorem**'e inecek — tensor-eval intrinsik olacağından $\omega(X,Y) \to \iota_X\iota_Y\omega$ otomatik türecek ve musical-inverse bilinear bağlantı $\pi(df, dg) = \omega(X_f, X_g)$'yi doğrulayacak. Bu notebook'un "tightened" bir revizyonu o zaman açılır.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Kurulum

- $f, g$ — fonksiyon (0-form).
- $\omega$ — symplectic 2-form (Graded(2)).
- $X_f, X_g$ — Hamiltonian vector fields (derece-0 Derivation).
- `poisson_fg` (`{f,g}`) ve `omega_XfXg` (`ω(X_f,X_g)`) — intrinsik node'umuz olmadığı için **isimli Symbol** olarak bildiriyoruz; problem metninin tanımsal özdeşlemelerini aşağıda aksiyom olarak ekliyoruz.
- Her iki composite (Poisson bracket ve 2-form eval) klasik Poisson hesabında **derece 0**: değerleri skalar fonksiyon.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.core.expr import Expr, Integer, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
g = Symbol("g")
reg.declare(f, Graded(degree=0))
reg.declare(g, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)
X_g = Derivation("X_g", degree=0)

# Named composites — system treats them as scalar (degree-0) symbols
# and the axioms below define what they equal.
poisson_fg = Symbol("{f,g}")
omega_XfXg = Symbol("ω(X_f,X_g)")
reg.declare(poisson_fg, Graded(degree=0))
reg.declare(omega_XfXg, Graded(degree=0))

print("f, g           :", f, g)
print("ω              :", omega)
print("X_f, X_g       :", X_f, X_g)
print("poisson_fg     :", poisson_fg)
print("omega_XfXg     :", omega_XfXg)

f, g           : f g
ω              : ω
X_f, X_g       : X_f X_g
poisson_fg     : {f,g}
omega_XfXg     : ω(X_f,X_g)


## 2. Dört problem-aksiyomu

Hepsi "LHS kalıbını gördüm, RHS ile değiştir" şeklinde birer `Definition`:

- **A1** — Poisson bracket tanımı: $\{f,g\} = \omega(X_f, X_g)$. Problem metninin $\pi = \omega^{-1}$ varsayımının doğrudan sonucu.
- **A2** — 2-form evaluation konvansiyonu: $\omega(X_f, X_g) = \iota_{X_f}\iota_{X_g}\omega$. Sistemimizdeki intrinsik boşluğu kapatıyor.
- **A3** — Hamiltonian defining relation (g): $\iota_{X_g}\omega = dg$. 2a'daki A2'nin $g$ muadili.
- **A4** — Hamiltonian defining relation (f): $\iota_{X_f}\omega = df$. Bu ispatın $X_f(g)$ kapanışı pairing'le direkt düştüğü için **teknik olarak gerekli değil**, ama problem verilenlerini tam yansıtmak için ekliyoruz.

In [3]:
class PoissonDef(Definition):
    """A1: {f,g} = ω(X_f, X_g)."""
    name = "{f,g} = ω(X_f, X_g)"

    def matches(self, expr):
        return expr == poisson_fg

    def rewrite(self, expr):
        return omega_XfXg


class FormEvalDef(Definition):
    """A2: ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω."""
    name = "ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω"

    def matches(self, expr):
        return expr == omega_XfXg

    def rewrite(self, expr):
        return Act(interior(X_f), Act(interior(X_g), omega))


class IotaXgOmegaIsDg(Definition):
    """A3: ι_{X_g} ω = dg."""
    name = "ι_{X_g} ω = dg"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_g:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, g)


class IotaXfOmegaIsDf(Definition):
    """A4: ι_{X_f} ω = df (completeness; not needed for this proof)."""
    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


for axiom in (PoissonDef, FormEvalDef, IotaXgOmegaIsDg, IotaXfOmegaIsDf):
    print("-", axiom.name)

- {f,g} = ω(X_f, X_g)
- ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω
- ι_{X_g} ω = dg
- ι_{X_f} ω = df


## 3. Engine

`default_engine` zaten pairing kuralını (**$\iota_X(df) = X(f)$**) taşıyor. Dört problem-aksiyomunu üstüne ekliyoruz.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(PoissonDef())
engine.register(FormEvalDef())
engine.register(IotaXgOmegaIsDg())
engine.register(IotaXfOmegaIsDf())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - {f,g} = ω(X_f, X_g)
 - ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω
 - ι_{X_g} ω = dg
 - ι_{X_f} ω = df


## 4. Hedef ve ispat

Zincirin uçtan uca eşitliği: $\{f, g\} = X_f(g)$. Engine bu obstruction üzerinde definition'ları fix-point'e kadar açacak; bottom-up fire ettiği için sıralama:
1. A1 — `{f,g} → ω(X_f,X_g)`
2. A2 — `ω(X_f,X_g) → ι_{X_f}(ι_{X_g}(ω))`
3. A3 — içten dışa: `ι_{X_g}(ω) → dg`
4. Pairing (yerleşik) — `ι_{X_f}(dg) → X_f(g)`
5. Simplify — `X_f(g) + (−X_f(g)) → 0`

In [5]:
lhs = poisson_fg
rhs = Act(X_f, g)

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nKAPANDI — {len(chain)} adım.")

LHS: {f,g}
RHS: X_f(g)

KAPANDI — 5 adım.


## 5. İspat zinciri — LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

\begin{align*}
{f,g} &\to \omega(X_f,X_g) && \text{[{f,g} = \ensuremath{\omega}(X\_f, X\_g)]\,(axiom)}\;\text{--- apply axiom: {f,g} = \ensuremath{\omega}(X\_f, X\_g)} \\
\omega(X_f,X_g) &\to \iota_{X_f}\!\left(\iota_{X_g}\!\left(\omega\right)\right) && \text{[\ensuremath{\omega}(X\_f, X\_g) = \ensuremath{\iota}\_{X\_f} \ensuremath{\iota}\_{X\_g} \ensuremath{\omega}]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\omega}(X\_f, X\_g) = \ensuremath{\iota}\_{X\_f} \ensuremath{\iota}\_{X\_g} \ensuremath{\omega}} \\
\iota_{X_g}\!\left(\omega\right) &\to d\!\left(g\right) && \text{[\ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg} \\
\iota_{X_f}\!\left(d\!\left(g\right)\right) &\to X_f\!\left(g\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
X_f\!\left(g\right) - X_f\!\left(g\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{align*}

## 6. Adım adım

Bu zincir aslında problem metnindeki beş-terim zincirinin kendisi — ara ifadeler doğal olarak ara değerler olarak çıkıyor.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" ↦  {step.after}")
    print()

[1] {f,g} = ω(X_f, X_g) [axiom]
    {f,g}
 ↦  ω(X_f,X_g)

[2] ω(X_f, X_g) = ι_{X_f} ι_{X_g} ω [axiom]
    ω(X_f,X_g)
 ↦  ι_X_f(ι_X_g(ω))

[3] ι_{X_g} ω = dg [axiom]
    ι_X_g(ω)
 ↦  d(g)

[4] ι_X(df) = X(f) [axiom]
    ι_X_f(d(g))
 ↦  X_f(g)

[5] simplify 
    (X_f(g) + (-X_f(g)))
 ↦  0



## 7. Zinciri problem-metni formatında yeniden göster

Beş terimi problem metnindeki sırada listeliyoruz. Her ara terim, zincirdeki bir ara ifadenin adı:

In [8]:
chain_terms = [
    (poisson_fg,                              "problem definition"),
    (omega_XfXg,                              "A1: {f,g} = ω(X_f, X_g)"),
    (Act(interior(X_f), Act(interior(X_g), omega)),
                                              "A2: ω(X_f, X_g) = ι_{X_f}ι_{X_g}ω"),
    (Act(interior(X_f), Act(d, g)),           "A3: ι_{X_g}ω = dg"),
    (Act(X_f, g),                             "built-in: ι_X(df) = X(f)"),
]

print("{f,g}")
for term, reason in chain_terms[1:]:
    print(f"   =  {term}   [{reason}]")

{f,g}
   =  ω(X_f,X_g)   [A1: {f,g} = ω(X_f, X_g)]
   =  ι_X_f(ι_X_g(ω))   [A2: ω(X_f, X_g) = ι_{X_f}ι_{X_g}ω]
   =  ι_X_f(d(g))   [A3: ι_{X_g}ω = dg]
   =  X_f(g)   [built-in: ι_X(df) = X(f)]


## Sonuç

$$\boxed{\;\{f, g\} = X_f(g)\;}$$

kapandı — 5-adımlık zincirde. Problem metnindeki beş-terimli gösterim, zincirdeki ara değerlerin tam olarak okunması.

Notebook'taki aksiyomların statüsü:

- **A3, A4** — gerçek matematiksel verilen (Hamiltonian defining relation); Faz 12'de de aksiyom kalmaya devam edecek.
- **A1, A2** — şu anda tanımsal aksiyom; Faz 12 landing ettiğinde intrinsik tensor-eval altyapısıyla **theorem**'e inecekler.
- **Pairing $\iota_X(df) = X(f)$** — zaten yerleşik; değişmez.

Bu zincir, 2a'daki $\mathcal{L}_{X_f}\omega = 0$ sonucuyla birlikte Poisson bracket'in temel hesap kimliklerini kapatır (antisimetri, Leibniz, Jacobi zinciri gibi sonraki kimlikler benzer paternle — ek aksiyom seti + aynı engine).